In [12]:
# Preprocessing script for HistorianEvents dataset (raw source → preprocessed parquets)
# Source: DATA/HistorianEvents (only the 5 UUID folders containing our alarm tags)
# Preprocessing steps: time offset (-1.5h), trip period removal, deduplication

import glob
import shutil
import pandas as pd
import numpy as np
from pathlib import Path

# ── Configuration ───────────────────────────────────────────────────
PROJECT_ROOT = Path('/home/h604827/ControlActions')
INPUT_DIR  = PROJECT_ROOT / 'DATA/HistorianEvents'
OUTPUT_DIR = PROJECT_ROOT / 'DATA/25_tags_events_preprocessed'
TRIP_CSV   = PROJECT_ROOT / 'DATA/Final_List_Trip_Duration.csv'
TIME_OFFSET = pd.Timedelta(hours=1.5)
DEDUP_COLS  = ['VT_Start', 'Source', 'ConditionName', 'Description']
CHUNK_SIZE  = 100_000

# Only process the 5 UUID folders that contain our alarm tags
TARGET_UUIDS = [
    '02e8226b-6cd8-43a3-b479-05a8dc9467bf',  # 03PIC_1620, 03FIC_1668, 03PI_1655, 03TIC_1635, 03LIC_1608, 03LIC_1619
    '51fda50f-24a3-4632-9803-7ec34744c34b',  # 03TIC_1009, 03PIC_1023, 03LIC_1016, 03TIC_1023
    '11266679-c683-445d-a1bf-29736e9e408f',  # 03LIC_1071, 03PIC_1104, 03TI_1081
    '73601713-e81f-4f46-8631-062c4dcd22dc',  # 03PIC_1013, 03TIC_1145, 03PI_1814
    '2c4de258-8026-4b49-8cfc-4a377becc808',  # 03TIC_1745A
]

# Clear old preprocessed files and recreate directory
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load trip periods (once) ────────────────────────────────────────
trip_data = pd.read_csv(TRIP_CSV)
trip_data['Stop Date']  = pd.to_datetime(trip_data['Stop Date'])
trip_data['Start Date'] = pd.to_datetime(trip_data['Start Date'])
stop_dates  = trip_data['Stop Date'].values
start_dates = trip_data['Start Date'].values

# ── Process target UUID folders ────────────────────────────────────
summary = []

for uuid_name in TARGET_UUIDS:
    uuid_dir = INPUT_DIR / uuid_name

    # Find all _E.parquet part files under this UUID folder
    parquet_files = glob.glob(
        str(uuid_dir / '2021011814_2025062803_E.parquet' / 'S=*' / '*.parquet')
    )
    if not parquet_files:
        print(f"  [{uuid_name[:8]}] No _E parquet files found – skipping")
        continue

    # 1. Read and concatenate all part files
    dfs = [pd.read_parquet(f) for f in parquet_files]
    df = pd.concat(dfs, ignore_index=True)
    n_raw = len(df)

    # 2. Adjust VT_Start by -1.5 hours
    if 'VT_Start' in df.columns:
        df['VT_Start'] = pd.to_datetime(df['VT_Start'], format='mixed') - TIME_OFFSET
    else:
        print(f"  [{uuid_name[:8]}] WARNING: VT_Start column not found – skipping time offset")

    df = df.sort_values('VT_Start').reset_index(drop=True)

    # 3. Remove events during trip periods (chunked broadcasting)
    event_times = df['VT_Start'].values
    in_trips = np.zeros(len(event_times), dtype=bool)

    for i in range(0, len(event_times), CHUNK_SIZE):
        chunk = event_times[i:i + CHUNK_SIZE]
        in_range = (chunk[:, None] >= stop_dates) & (chunk[:, None] <= start_dates)
        in_trips[i:i + CHUNK_SIZE] = in_range.any(axis=1)

    n_trip = int(in_trips.sum())
    df = df[~in_trips].reset_index(drop=True)

    # 4. Deduplicate rows
    n_pre_dedup = len(df)
    df = df.groupby(DEDUP_COLS, sort=False).first().reset_index()
    df = df.sort_values('VT_Start').reset_index(drop=True)
    n_deduped = n_pre_dedup - len(df)

    # 5. Save preprocessed parquet
    out_path = OUTPUT_DIR / f'{uuid_name}.parquet'
    df.to_parquet(out_path, index=False)

    summary.append({
        'uuid': uuid_name,
        'raw_rows': n_raw,
        'trip_removed': n_trip,
        'duplicates_removed': n_deduped,
        'final_rows': len(df),
        'sources': df['Source'].nunique() if 'Source' in df.columns else None
    })
    print(f"  [{uuid_name[:8]}] {n_raw:,} → {len(df):,} rows "
          f"(trips: -{n_trip:,}, dupes: -{n_deduped:,}) → {out_path.name}")

# ── Summary ────────────────────────────────────────────────────────
summary_df = pd.DataFrame(summary)
print(f"\n{'='*70}")
print(f"Preprocessed {len(summary_df)} UUID folders → {OUTPUT_DIR}/")
print(f"Total: {summary_df['raw_rows'].sum():,} raw → {summary_df['final_rows'].sum():,} final rows")
print(summary_df.to_string(index=False))

  [02e8226b] 197,150 → 136,265 rows (trips: -28,563, dupes: -32,322) → 02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet
  [51fda50f] 142,167 → 70,213 rows (trips: -18,656, dupes: -53,298) → 51fda50f-24a3-4632-9803-7ec34744c34b.parquet
  [11266679] 138,249 → 43,675 rows (trips: -70,011, dupes: -24,563) → 11266679-c683-445d-a1bf-29736e9e408f.parquet
  [73601713] 1,051,748 → 906,230 rows (trips: -96,538, dupes: -48,980) → 73601713-e81f-4f46-8631-062c4dcd22dc.parquet
  [2c4de258] 124,569 → 68,510 rows (trips: -5,070, dupes: -50,989) → 2c4de258-8026-4b49-8cfc-4a377becc808.parquet

Preprocessed 5 UUID folders → /home/h604827/ControlActions/DATA/25_tags_events_preprocessed/
Total: 1,653,883 raw → 1,224,893 final rows
                                uuid  raw_rows  trip_removed  duplicates_removed  final_rows  sources
02e8226b-6cd8-43a3-b479-05a8dc9467bf    197150         28563               32322      136265       50
51fda50f-24a3-4632-9803-7ec34744c34b    142167         18656               53298

In [25]:
import pandas as pd

df = pd.read_parquet('/home/h604827/ControlActions/DATA/HistorianEvents/2c4de258-8026-4b49-8cfc-4a377becc808/2021011814_2025062803_EAL.parquet/S=R520-ESVT/part-00000-1f634fef-2ab8-4dab-8754-960209c71d1e.c000.snappy.parquet')

df
# df['TagName'].value_counts()

,ServerName,EventID,IntervalIdentifier,Priority,Parameter,FromValue,ToValue,Value,Description,ACTION,...,VT_Start,TagID,EventTypeID,VT_End,LastEvent,NextEvent,rowid,Seconds,PrevSeconds,H
0,R520-ESVT,74388940,CMDDIS,16,None,None,CLOSE,CLOSE,3F101 ESD COIL VENT VALV,None,...,2024-12-04 18:34:41.4044,03EDPV_1795,1,None,0,2,1,35248,6,2024_12_04_18
1,R520-ESVT,74394968,CMDDIS,16,None,None,CLOSE,CLOSE,3F101 ESD COIL VENT VALV,OK,...,2024-12-05 04:22:09.4056,03EDPV_1795,2,None,1,0,2,6,35248,2024_12_05_04
2,R520-ESVT,69247511,CMDFAIL,16,None,None,CLOSE,CLOSE,3F101 ESD COIL VENT VALV,None,...,2024-02-22 11:06:14.1521,03EDPV_1795,1,None,2,2,5,8,37759264,2024_02_22_11
3,R520-ESVT,69247514,CMDFAIL,16,None,None,MOVING,MOVING,3F101 ESD COIL VENT VALV,OK,...,2024-02-22 11:06:22.1526,03EDPV_1795,2,None,1,1,6,642278,8,2024_02_22_11
4,R520-ESVT,74388886,CMDFAIL,16,None,None,CLOSE,CLOSE,3F101 ESD COIL VENT VALV,None,...,2024-12-04 18:34:26.4026,03EDPV_1795,1,None,2,2,9,35263,24095002,2024_12_04_18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,R520-ESVT,77085085,PVHI,16,None,None,251.877,251.877,3F101 O/L TEMP CNTRL,OK,...,2025-01-10 11:16:35.1069,03TIC_1745A,2,None,1,1,343,10456197,17,2025_01_10_11
245,R520-ESVT,78670641,PVHI,16,None,None,270.005,270.005,3F101 O/L TEMP CNTRL,None,...,2025-05-11 11:46:32.628,03TIC_1745A,1,None,2,2,344,133,10456197,2025_05_11_11
246,R520-ESVT,78670683,PVHI,16,None,None,266.498,266.498,3F101 O/L TEMP CNTRL,OK,...,2025-05-11 11:48:45.5896,03TIC_1745A,2,None,1,1,345,782654,133,2025_05_11_11
247,R520-ESVT,78912486,PVHI,16,None,None,270.038,270.038,3F101 O/L TEMP CNTRL,None,...,2025-05-20 13:12:59.3339,03TIC_1745A,1,None,2,2,346,218,782654,2025_05_20_13


In [14]:
data_path = '/home/h604827/ControlActions/DATA/25_tags_events_preprocessed'

alarm_tags = [
    ['03PIC_1620', 'PVHI'],
    ['03FIC_1668', 'PVHI'],
    ['03PI_1655', 'PVHI'],
    ['03TIC_1635', 'PVHI'],
    ['03TIC_1009', 'PVHI'],
    ['03TIC_1745A', 'PVHI'],
    ['03LIC_1608', 'PVHI'],
    ['03PIC_1013', 'PVLO'],
    ['03LIC_1608', 'PVLO'],
    ['03PIC_1023', 'PVLO'],
    ['03TIC_1145', 'PVLO'],
    ['03LIC_1071', 'PVHI'],
    ['03LIC_1016', 'PVHI'],
    ['03PIC_1104', 'PVHI'],
    ['03TIC_1009', 'PVLO'],
    ['03LIC_1619', 'PVLO'],
    ['03TIC_1635', 'PVLO'],
    ['03PIC_1023', 'PVHI'],
    ['03LIC_1619', 'PVHI'],
    ['03PI_1814', 'PVHI'],
    ['03TIC_1023', 'PVLO'],
    ['03TIC_1023', 'PVHI'],
    ['03LIC_1016', 'PVLO'],
    ['03LIC_1071', 'PVLO'],
    ['03TI_1081', 'PVHI'],
]

In [15]:
import pandas as pd
from pathlib import Path

data_path = Path('/home/h604827/ControlActions/DATA/25_tags_events_preprocessed')
parquet_files = sorted(data_path.glob('*.parquet'))
events_csv_path = '/home/h604827/ControlActions/DATA/trip_filtered_events_dedup.csv'

# For each parquet file, check which alarm_tag entries it contains
results = []
for pf in parquet_files:
    df_tmp = pd.read_parquet(pf, columns=['Source', 'ConditionName'])
    for tag, condition in alarm_tags:
        mask = (df_tmp['Source'] == tag) & (df_tmp['ConditionName'] == condition)
        count = mask.sum()
        if count > 0:
            results.append({'file': pf.name, 'tag': tag, 'condition': condition, 'event_count': count})

# Check trip_filtered_events_dedup.csv for remaining tags
events_df = pd.read_csv(events_csv_path, low_memory=False)
for tag, condition in alarm_tags:
    mask = (events_df['Source'] == tag) & (events_df['ConditionName'] == condition)
    count = mask.sum()
    if count > 0:
        results.append({'file': 'trip_filtered_events_dedup.csv', 'tag': tag, 'condition': condition, 'event_count': count})

results_df = pd.DataFrame(results)
print(f"Total matches: {len(results_df)}")
print(f"\nAlarm tags with matches: {results_df[['tag','condition']].drop_duplicates().shape[0]} / {len(alarm_tags)}")
print()
results_df.pivot_table(index=['tag', 'condition'], columns='file', values='event_count', fill_value=0)

Total matches: 40

Alarm tags with matches: 25 / 25



file                   02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet  \
tag         condition                                                 
03FIC_1668  PVHI                                               19.0   
03LIC_1016  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1071  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1608  PVHI                                               22.0   
            PVLO                                               63.0   
03LIC_1619  PVHI                                             2550.0   
            PVLO                                             2027.0   
03PIC_1013  PVLO                                                0.0   
03PIC_1023  PVHI                                                0.0   
            PVLO                                                0.0   
03PIC_1104  PVHI                                                0.0   
03PIC_1620  PVHI                                               19.0   
03PI_1655   PVHI                                               29.0   
03PI_1814   PVHI                                                0.0   
03TIC_1009  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1023  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1145  PVLO                                                0.0   
03TIC_1635  PVHI                                               17.0   
            PVLO                                             1629.0   
03TIC_1745A PVHI                                                0.0   
03TI_1081   PVHI                                                0.0   

file                   11266679-c683-445d-a1bf-29736e9e408f.parquet  \
tag         condition                                                 
03FIC_1668  PVHI                                                0.0   
03LIC_1016  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1071  PVHI                                              499.0   
            PVLO                                             4518.0   
03LIC_1608  PVHI                                                0.0   
            PVLO                                                0.0   
03LIC_1619  PVHI                                                0.0   
            PVLO                                                0.0   
03PIC_1013  PVLO                                                0.0   
03PIC_1023  PVHI                                                0.0   
            PVLO                                                0.0   
03PIC_1104  PVHI                                              928.0   
03PIC_1620  PVHI                                                0.0   
03PI_1655   PVHI                                                0.0   
03PI_1814   PVHI                                                0.0   
03TIC_1009  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1023  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1145  PVLO                                                0.0   
03TIC_1635  PVHI                                                0.0   
            PVLO                                                0.0   
03TIC_1745A PVHI                                                0.0   
03TI_1081   PVHI                                             1681.0   

file                   2c4de258-8026-4b49-8cfc-4a377becc808.parquet  \
tag         condition                                                 
03FIC_1668  PVHI      

In [16]:
# Compare presence in parquet files (from HistorianEvents) vs CSV for each alarm tag
# After preprocessing from HistorianEvents, ALL 25 tags should be in parquets.
# This cell verifies counts match between the two sources for overlapping tags.

parquet_results = results_df[results_df['file'] != 'trip_filtered_events_dedup.csv']
csv_results = results_df[results_df['file'] == 'trip_filtered_events_dedup.csv']

print(f"{'Tag':<15} | {'Cond':<4} | {'Parquet File':<45} | {'Parquet':>8} | {'CSV':>8} | Status")
print("-" * 110)

for tag, condition in alarm_tags:
    # Check parquet
    pq_match = parquet_results[(parquet_results['tag'] == tag) & (parquet_results['condition'] == condition)]
    pq_file = pq_match['file'].values[0] if len(pq_match) > 0 else '—'
    pq_count = pq_match['event_count'].values[0] if len(pq_match) > 0 else 0
    
    # Check CSV
    csv_match = csv_results[(csv_results['tag'] == tag) & (csv_results['condition'] == condition)]
    csv_count = csv_match['event_count'].values[0] if len(csv_match) > 0 else 0
    
    # Determine status
    if pq_count > 0 and csv_count > 0:
        if pq_count == csv_count:
            status = '✓ MATCH'
        else:
            status = f'✗ MISMATCH (diff={pq_count - csv_count:+d})'
    elif pq_count > 0:
        status = '[parquet only]'
    elif csv_count > 0:
        status = '[csv only]'
    else:
        status = '✗ MISSING'
    
    print(f"{tag:<15} | {condition:<4} | {pq_file:<45} | {pq_count:>8,} | {csv_count:>8,} | {status}")

print(f"\n{'='*110}")
in_both_match = sum(1 for t, c in alarm_tags 
    if len(parquet_results[(parquet_results['tag']==t) & (parquet_results['condition']==c)]) > 0
    and len(csv_results[(csv_results['tag']==t) & (csv_results['condition']==c)]) > 0
    and parquet_results[(parquet_results['tag']==t) & (parquet_results['condition']==c)]['event_count'].values[0]
        == csv_results[(csv_results['tag']==t) & (csv_results['condition']==c)]['event_count'].values[0])
in_both_mismatch = sum(1 for t, c in alarm_tags 
    if len(parquet_results[(parquet_results['tag']==t) & (parquet_results['condition']==c)]) > 0
    and len(csv_results[(csv_results['tag']==t) & (csv_results['condition']==c)]) > 0
    and parquet_results[(parquet_results['tag']==t) & (parquet_results['condition']==c)]['event_count'].values[0]
        != csv_results[(csv_results['tag']==t) & (csv_results['condition']==c)]['event_count'].values[0])
in_parquet_only = sum(1 for t, c in alarm_tags 
    if len(parquet_results[(parquet_results['tag']==t) & (parquet_results['condition']==c)]) > 0
    and len(csv_results[(csv_results['tag']==t) & (csv_results['condition']==c)]) == 0)
in_csv_only = sum(1 for t, c in alarm_tags 
    if len(parquet_results[(parquet_results['tag']==t) & (parquet_results['condition']==c)]) == 0
    and len(csv_results[(csv_results['tag']==t) & (csv_results['condition']==c)]) > 0)
print(f"In BOTH (matching): {in_both_match} | In BOTH (mismatch): {in_both_mismatch} | Parquet only: {in_parquet_only} | CSV only: {in_csv_only}")

Tag             | Cond | Parquet File                                  |  Parquet |      CSV | Status
--------------------------------------------------------------------------------------------------------------
03PIC_1620      | PVHI | 02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet  |       19 |        0 | [parquet only]
03FIC_1668      | PVHI | 02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet  |       19 |        0 | [parquet only]
03PI_1655       | PVHI | 02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet  |       29 |        0 | [parquet only]
03TIC_1635      | PVHI | 02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet  |       17 |        0 | [parquet only]
03TIC_1009      | PVHI | 51fda50f-24a3-4632-9803-7ec34744c34b.parquet  |       10 |       10 | ✓ MATCH
03TIC_1745A     | PVHI | 2c4de258-8026-4b49-8cfc-4a377becc808.parquet  |       31 |        0 | [parquet only]
03LIC_1608      | PVHI | 02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet  |       22 |        0 | [parquet only]
03PIC_1013      | PVLO |

In [10]:
df = pd.read_parquet('/home/h604827/ControlActions/DATA/25_tags_events_preprocessed/02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet')
df[(df['Source'] == '03FIC_1668') & (df['Category'] == 1)]

,VT_Start,Source,ConditionName,Description,Action,Actor,AreaName,AlarmLimit,Block,Category,...,ShelvedReason,SourceParameter,Station,Time,TransactionID,Units,Value,H,TagID,AlarmStatus
2037,2021-10-30 06:19:33.853300,03FIC_1668,PVLO,3E113A/B GLYCOL FLW CTRL,None,None,1O,3000.0,None,1,...,None,None,None,132800339738533000,31613383,None,2992.30,2021_10_30_07,03FIC_1668,ENABLE
2051,2021-10-30 07:04:14.352200,03FIC_1668,PVLO,3E113A/B GLYCOL FLW CTRL,OK,None,1O,3000.0,None,1,...,None,None,None,132800366543522000,31613383,None,3395.64,2021_10_30_08,03FIC_1668,ENABLE
3164,2021-11-24 19:40:05.405000,03FIC_1668,PVLO,3E113A/B GLYCOL FLW CTRL,None,None,1O,3000.0,None,1,...,None,None,None,132822420054050000,31965033,None,2977.44,2021_11_24_21,03FIC_1668,ENABLE
3565,2021-11-30 09:22:48.102500,03FIC_1668,PVLO,3E113A/B GLYCOL FLW CTRL,None,None,1O,3000.0,None,1,...,None,None,None,132827233681025000,32351778,None,-143.041,2021_11_30_10,03FIC_1668,ENABLE
3574,2021-11-30 09:30:20.601900,03FIC_1668,PVLO,3E113A/B GLYCOL FLW CTRL,None,None,1O,3000.0,None,1,...,None,None,None,132827238206019000,32351778,None,-143.041,2021_11_30_11,03FIC_1668,ENABLE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93695,2025-01-08 21:47:38.356300,03FIC_1668,PVLO,3E113A/B GLYCOL FLW CTRL,OK,None,1O,3000.0,None,1,...,None,None,None,133808320583563000,77074714,None,3380.79,2025_01_08_23,03FIC_1668,ENABLE
94439,2025-03-06 11:14:28.864700,03FIC_1668,PVLO,3E113A/B GLYCOL FLW CTRL,None,None,1O,3000.0,None,1,...,None,None,None,133857188688647000,77443652,None,2996.87,2025_03_06_12,03FIC_1668,ENABLE
94440,2025-03-06 11:14:42.924900,03FIC_1668,PVLO,3E113A/B GLYCOL FLW CTRL,OK,None,1O,3000.0,None,1,...,None,None,None,133857188829249000,77443652,None,3375.07,2025_03_06_12,03FIC_1668,ENABLE
130116,2025-05-03 18:36:45.006500,03FIC_1668,PVLO,3E113A/B GLYCOL FLW CTRL,None,None,1O,3000.0,None,1,...,None,None,None,133907566050065000,78486316,None,2779.77,2025_05_03_20,03FIC_1668,ENABLE


In [11]:
import pandas as pd
from pathlib import Path

data_path = Path('/home/h604827/ControlActions/DATA/25_tags_events_preprocessed')
events_csv_path = '/home/h604827/ControlActions/DATA/trip_filtered_events_dedup.csv'

# Load CSV (already in memory as events_df, but ensure it's available)
events_df = pd.read_csv(events_csv_path, low_memory=False)
events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])

# Build a lookup: which parquet file has each tag
parquet_files = sorted(data_path.glob('*.parquet'))
tag_parquet_map = {}  # (tag, condition) -> parquet file path
for pf in parquet_files:
    df_tmp = pd.read_parquet(pf, columns=['Source', 'ConditionName'])
    sources = df_tmp['Source'].unique()
    conditions = df_tmp['ConditionName'].unique()
    for tag, condition in alarm_tags:
        if tag in sources:
            # Verify this condition exists in the file
            mask = (df_tmp['Source'] == tag) & (df_tmp['ConditionName'] == condition)
            if mask.any():
                tag_parquet_map[(tag, condition)] = pf

def load_tag_events(tag, condition):
    """Load alarm-relevant events for a tag/condition from best source."""
    if (tag, condition) in tag_parquet_map:
        pf = tag_parquet_map[(tag, condition)]
        df = pd.read_parquet(pf)
        df['VT_Start'] = pd.to_datetime(df['VT_Start'])
        df = df[(df['Source'] == tag) & (df['ConditionName'] == condition)]
    else:
        df = events_df[(events_df['Source'] == tag) & (events_df['ConditionName'] == condition)].copy()
    
    # Filter Category == 1
    df = df[df['Category'] == 1].copy()
    df = df.sort_values('VT_Start').reset_index(drop=True)
    return df

def extract_alarms(tag, condition):
    """Extract alarm episodes (start→end pairs) and log orphans."""
    df = load_tag_events(tag, condition)
    
    # Identify starts and ends
    is_start = df['Action'].isna() | (df['Action'] == '')
    is_end = df['Action'] == 'OK'
    
    starts = df[is_start].reset_index(drop=True)
    ends = df[is_end].reset_index(drop=True)
    
    alarms = []
    orphan_starts = []
    orphan_ends = []
    
    # Sequential pairing: walk through events in time order
    events_ordered = df[is_start | is_end].copy()
    events_ordered['is_start'] = events_ordered['Action'].isna() | (events_ordered['Action'] == '')
    events_ordered = events_ordered.sort_values('VT_Start').reset_index(drop=True)
    
    pending_start = None
    for idx, row in events_ordered.iterrows():
        if row['is_start']:
            if pending_start is not None:
                # Previous start had no end → orphan start
                orphan_starts.append(pending_start)
            pending_start = row
        else:  # is_end (Action == 'OK')
            if pending_start is not None:
                # Matched pair
                alarms.append({
                    'tag': tag,
                    'condition': condition,
                    'alarm_start': pending_start['VT_Start'],
                    'alarm_end': row['VT_Start'],
                    'duration_minutes': (row['VT_Start'] - pending_start['VT_Start']).total_seconds() / 60
                })
                pending_start = None
            else:
                # End with no preceding start → orphan end
                orphan_ends.append(row)
    
    # If there's a remaining pending start at the end
    if pending_start is not None:
        orphan_starts.append(pending_start)
    
    return alarms, orphan_starts, orphan_ends

# ── Extract alarms for all 25 entries ──────────────────────────────
all_alarms = []
all_orphan_starts = []
all_orphan_ends = []

print(f"{'Tag':<15} | {'Cond':<4} | {'Alarms':>6} | {'Orphan Starts':>13} | {'Orphan Ends':>11}")
print("-" * 70)

for tag, condition in alarm_tags:
    alarms, orphan_starts, orphan_ends = extract_alarms(tag, condition)
    all_alarms.extend(alarms)
    
    for os_row in orphan_starts:
        all_orphan_starts.append({'tag': tag, 'condition': condition, 'timestamp': os_row['VT_Start']})
    for oe_row in orphan_ends:
        all_orphan_ends.append({'tag': tag, 'condition': condition, 'timestamp': oe_row['VT_Start']})
    
    print(f"{tag:<15} | {condition:<4} | {len(alarms):>6} | {len(orphan_starts):>13} | {len(orphan_ends):>11}")

print("-" * 70)
print(f"{'TOTAL':<15} | {'':4} | {len(all_alarms):>6} | {len(all_orphan_starts):>13} | {len(all_orphan_ends):>11}")

# Create DataFrames
alarms_df = pd.DataFrame(all_alarms)
orphan_starts_df = pd.DataFrame(all_orphan_starts) if all_orphan_starts else pd.DataFrame(columns=['tag','condition','timestamp'])
orphan_ends_df = pd.DataFrame(all_orphan_ends) if all_orphan_ends else pd.DataFrame(columns=['tag','condition','timestamp'])

print(f"\n\nAlarms DataFrame shape: {alarms_df.shape}")
print(f"Orphan starts: {len(orphan_starts_df)}")
print(f"Orphan ends: {len(orphan_ends_df)}")
alarms_df.head(10)

Tag             | Cond | Alarms | Orphan Starts | Orphan Ends
----------------------------------------------------------------------
03PIC_1620      | PVHI |      6 |             0 |           0
03FIC_1668      | PVHI |      7 |             0 |           0
03PI_1655       | PVHI |      9 |             0 |           0
03TIC_1635      | PVHI |      5 |             0 |           0
03TIC_1009      | PVHI |      3 |             0 |           0
03TIC_1745A     | PVHI |      9 |             0 |           0
03LIC_1608      | PVHI |      7 |             0 |           0
03PIC_1013      | PVLO |     23 |             3 |           0
03LIC_1608      | PVLO |     19 |             0 |           0
03PIC_1023      | PVLO |     18 |             5 |           0
03TIC_1145      | PVLO |     67 |             2 |           0
03LIC_1071      | PVHI |    184 |             1 |           1
03LIC_1016      | PVHI |    154 |             0 |           0
03PIC_1104      | PVHI |    276 |             0 |           0

,tag,condition,alarm_start,alarm_end,duration_minutes
0,03PIC_1620,PVHI,2021-10-15 07:10:19.205900,2021-10-15 07:10:37.205900,0.300000
1,03PIC_1620,PVHI,2021-11-30 09:12:19.601600,2021-11-30 09:13:21.603000,1.033357
2,03PIC_1620,PVHI,2021-12-07 03:21:49.752900,2021-12-07 03:22:21.753600,0.533345
3,03PIC_1620,PVHI,2021-12-07 13:29:57.302900,2021-12-07 13:31:36.307300,1.650073
4,03PIC_1620,PVHI,2023-01-30 08:35:25.603500,2023-01-30 08:36:17.603000,0.866658
5,03PIC_1620,PVHI,2023-11-06 00:50:50.203600,2023-11-06 00:51:34.202700,0.733318
6,03FIC_1668,PVHI,2021-12-07 02:28:58.902700,2021-12-07 02:29:05.903600,0.116682
7,03FIC_1668,PVHI,2024-01-06 10:44:02.103500,2024-01-06 10:44:40.102600,0.633318
8,03FIC_1668,PVHI,2024-04-28 08:45:14.752000,2024-04-28 08:46:11.753000,0.950017
9,03FIC_1668,PVHI,2024-04-28 08:46:35.753100,2024-04-28 08:47:11.751500,0.599973


In [27]:
# ── Compare alarm durations: _E source vs _EAL source ──────────────────────────
# _E source: alarms_df already extracted above (using Action=null→start, Action='OK'→end)
# _EAL source: extract using EventTypeID=1→start, EventTypeID=2→end, TagName, IntervalIdentifier

import glob
import numpy as np

EAL_INPUT_DIR = Path('/home/h604827/ControlActions/DATA/HistorianEvents')
EAL_TARGET_UUIDS = [
    '02e8226b-6cd8-43a3-b479-05a8dc9467bf',
    '51fda50f-24a3-4632-9803-7ec34744c34b',
    '11266679-c683-445d-a1bf-29736e9e408f',
    '73601713-e81f-4f46-8631-062c4dcd22dc',
    '2c4de258-8026-4b49-8cfc-4a377becc808',
]

# Load and combine EAL data from the 5 relevant UUID folders
eal_dfs = []
for uuid_name in EAL_TARGET_UUIDS:
    eal_dir = EAL_INPUT_DIR / uuid_name / '2021011814_2025062803_EAL.parquet'
    part_files = glob.glob(str(eal_dir / 'S=*' / '*.parquet'))
    if part_files:
        for f in part_files:
            df_tmp = pd.read_parquet(f, columns=['TagName', 'IntervalIdentifier', 'EventTypeID', 'VT_Start'])
            eal_dfs.append(df_tmp)

eal_all = pd.concat(eal_dfs, ignore_index=True)
eal_all['VT_Start'] = pd.to_datetime(eal_all['VT_Start'])
# Apply same -1.5h time offset as _E source
eal_all['VT_Start'] = eal_all['VT_Start'] - pd.Timedelta(hours=1.5)
eal_all = eal_all.sort_values('VT_Start').reset_index(drop=True)
print(f"EAL total rows loaded (before trip filter): {len(eal_all):,}")

# ── Remove events during trip periods (same logic as _E preprocessing) ──
TRIP_CSV = Path('/home/h604827/ControlActions/DATA/Final_List_Trip_Duration.csv')
trip_data = pd.read_csv(TRIP_CSV)
trip_data['Stop Date'] = pd.to_datetime(trip_data['Stop Date'])
trip_data['Start Date'] = pd.to_datetime(trip_data['Start Date'])
stop_dates = trip_data['Stop Date'].values
start_dates = trip_data['Start Date'].values

event_times = eal_all['VT_Start'].values
in_trips = np.zeros(len(event_times), dtype=bool)
CHUNK_SIZE = 100_000

for i in range(0, len(event_times), CHUNK_SIZE):
    chunk = event_times[i:i + CHUNK_SIZE]
    in_range = (chunk[:, None] >= stop_dates) & (chunk[:, None] <= start_dates)
    in_trips[i:i + CHUNK_SIZE] = in_range.any(axis=1)

n_trip_removed = int(in_trips.sum())
eal_all = eal_all[~in_trips].reset_index(drop=True)
print(f"EAL rows after trip filter: {len(eal_all):,} (removed {n_trip_removed:,} trip events)")

# Extract alarm episodes from EAL using EventTypeID
def extract_alarms_eal(tag, condition):
    """Extract alarm episodes from EAL data using EventTypeID=1 (start), EventTypeID=2 (end)."""
    df = eal_all[(eal_all['TagName'] == tag) & (eal_all['IntervalIdentifier'] == condition)].copy()
    df = df.sort_values('VT_Start').reset_index(drop=True)
    
    alarms = []
    orphan_starts = []
    orphan_ends = []
    
    pending_start = None
    for _, row in df.iterrows():
        if row['EventTypeID'] == 1:  # Alarm start
            if pending_start is not None:
                orphan_starts.append(pending_start)
            pending_start = row
        elif row['EventTypeID'] == 2:  # Alarm end
            if pending_start is not None:
                alarms.append({
                    'tag': tag,
                    'condition': condition,
                    'alarm_start': pending_start['VT_Start'],
                    'alarm_end': row['VT_Start'],
                    'duration_minutes': (row['VT_Start'] - pending_start['VT_Start']).total_seconds() / 60
                })
                pending_start = None
            else:
                orphan_ends.append(row)
    
    if pending_start is not None:
        orphan_starts.append(pending_start)
    
    return alarms, orphan_starts, orphan_ends

# Extract EAL alarms for all 25 tags
eal_alarms = []
print(f"\n{'Tag':<15} | {'Cond':<4} | {'EAL Alarms':>10} | {'E Alarms':>9} | {'Diff':>6}")
print("-" * 60)

for tag, condition in alarm_tags:
    alarms_eal, _, _ = extract_alarms_eal(tag, condition)
    eal_alarms.extend(alarms_eal)
    
    e_count = len(alarms_df[(alarms_df['tag'] == tag) & (alarms_df['condition'] == condition)])
    print(f"{tag:<15} | {condition:<4} | {len(alarms_eal):>10} | {e_count:>9} | {len(alarms_eal) - e_count:>+6}")

eal_alarms_df = pd.DataFrame(eal_alarms)
print(f"\n{'='*60}")
print(f"Total: EAL={len(eal_alarms_df):,} vs E={len(alarms_df):,}")

# ── Compare alarm durations per tag/condition ──────────────────────
print(f"\n\n{'='*80}")
print("ALARM DURATION COMPARISON (minutes): _E vs _EAL")
print(f"{'='*80}")
print(f"\n{'Tag':<15} | {'Cond':<4} | {'Source':<5} | {'Count':>6} | {'Mean':>8} | {'Median':>8} | {'Min':>8} | {'Max':>10}")
print("-" * 90)

for tag, condition in alarm_tags:
    e_eps = alarms_df[(alarms_df['tag'] == tag) & (alarms_df['condition'] == condition)]
    eal_eps = eal_alarms_df[(eal_alarms_df['tag'] == tag) & (eal_alarms_df['condition'] == condition)]
    
    if len(e_eps) > 0:
        print(f"{tag:<15} | {condition:<4} | {'E':<5} | {len(e_eps):>6} | {e_eps['duration_minutes'].mean():>8.1f} | {e_eps['duration_minutes'].median():>8.1f} | {e_eps['duration_minutes'].min():>8.1f} | {e_eps['duration_minutes'].max():>10.1f}")
    if len(eal_eps) > 0:
        print(f"{'':<15} | {'':<4} | {'EAL':<5} | {len(eal_eps):>6} | {eal_eps['duration_minutes'].mean():>8.1f} | {eal_eps['duration_minutes'].median():>8.1f} | {eal_eps['duration_minutes'].min():>8.1f} | {eal_eps['duration_minutes'].max():>10.1f}")
    print()

EAL total rows loaded (before trip filter): 115,261
EAL rows after trip filter: 95,230 (removed 20,031 trip events)

Tag             | Cond | EAL Alarms |  E Alarms |   Diff
------------------------------------------------------------
03PIC_1620      | PVHI |          6 |         6 |     +0
03FIC_1668      | PVHI |          4 |         7 |     -3
03PI_1655       | PVHI |          9 |         9 |     +0
03TIC_1635      | PVHI |          5 |         5 |     +0
03TIC_1009      | PVHI |          3 |         3 |     +0
03TIC_1745A     | PVHI |          9 |         9 |     +0
03LIC_1608      | PVHI |          7 |         7 |     +0
03PIC_1013      | PVLO |         23 |        23 |     +0
03LIC_1608      | PVLO |         19 |        19 |     +0
03PIC_1023      | PVLO |         18 |        18 |     +0
03TIC_1145      | PVLO |         34 |        67 |    -33
03LIC_1071      | PVHI |        165 |       184 |    -19
03LIC_1016      | PVHI |        154 |       154 |     +0
03PIC_1104      | PVHI |

In [30]:
eal_alarms_df.columns

Index(['tag', 'condition', 'alarm_start', 'alarm_end', 'duration_minutes'], dtype='object')

In [31]:
alarms_df[alarms_df['tag'] == '03FIC_1668']

,tag,condition,alarm_start,alarm_end,duration_minutes
6,03FIC_1668,PVHI,2021-12-07 02:28:58.902700,2021-12-07 02:29:05.903600,0.116682
7,03FIC_1668,PVHI,2024-01-06 10:44:02.103500,2024-01-06 10:44:40.102600,0.633318
8,03FIC_1668,PVHI,2024-04-28 08:45:14.752000,2024-04-28 08:46:11.753000,0.950017
9,03FIC_1668,PVHI,2024-04-28 08:46:35.753100,2024-04-28 08:47:11.751500,0.599973
10,03FIC_1668,PVHI,2024-12-26 09:41:06.205600,2024-12-26 09:41:11.615300,0.090162
11,03FIC_1668,PVHI,2024-12-26 09:41:14.354200,2024-12-26 09:41:16.461300,0.035118
12,03FIC_1668,PVHI,2024-12-26 09:42:43.596300,2024-12-26 09:42:48.471500,0.081253


In [32]:
# ── Filter alarm tags with more than 15 alarms ──────────────────────
alarm_counts = alarms_df.groupby(['tag', 'condition']).size().reset_index(name='alarm_count')
significant_alarms = alarm_counts[alarm_counts['alarm_count'] > 15]
print(f"Tags with >15 alarms: {len(significant_alarms)} / {len(alarm_counts)}")
print(significant_alarms.to_string(index=False))

# Filter alarms_df to only significant tags
significant_pairs = set(zip(significant_alarms['tag'], significant_alarms['condition']))
alarms_significant = alarms_df[
    alarms_df.apply(lambda r: (r['tag'], r['condition']) in significant_pairs, axis=1)
].copy().reset_index(drop=True)
print(f"\nTotal alarm episodes to analyze: {len(alarms_significant)}")

Tags with >15 alarms: 18 / 25
       tag condition  alarm_count
03LIC_1016      PVHI          154
03LIC_1016      PVLO         2265
03LIC_1071      PVHI          184
03LIC_1071      PVLO         1400
03LIC_1608      PVLO           19
03LIC_1619      PVHI          782
03LIC_1619      PVLO          665
03PIC_1013      PVLO           23
03PIC_1023      PVHI          333
03PIC_1023      PVLO           18
03PIC_1104      PVHI          276
 03PI_1814      PVHI          266
03TIC_1009      PVLO          122
03TIC_1023      PVHI          504
03TIC_1023      PVLO          865
03TIC_1145      PVLO           67
03TIC_1635      PVLO          469
 03TI_1081      PVHI          480

Total alarm episodes to analyze: 8892


In [33]:
# ── Extract control actions (CHANGE events) around each alarm episode ──────
# For each alarm tag, use ONLY the same data source where alarm events come from
# This ensures control actions are from the same tag group / process area

WINDOW_BEFORE = pd.Timedelta(minutes=30)
WINDOW_AFTER = pd.Timedelta(minutes=30)

# Pre-load CHANGE events per source file (cache to avoid re-reading)
print("Loading CHANGE events per source...")
change_events_by_source = {}

# From parquet files
for pf in parquet_files:
    df_tmp = pd.read_parquet(pf)
    df_tmp['VT_Start'] = pd.to_datetime(df_tmp['VT_Start'])
    changes = df_tmp[df_tmp['ConditionName'] == 'CHANGE'].copy()
    if len(changes) > 0:
        changes = changes.sort_values('VT_Start').reset_index(drop=True)
        change_events_by_source[pf.name] = changes
        print(f"  {pf.name}: {len(changes):,} CHANGE events")

# From CSV
csv_changes = events_df[events_df['ConditionName'] == 'CHANGE'].copy()
csv_changes = csv_changes.sort_values('VT_Start').reset_index(drop=True)
change_events_by_source['trip_filtered_events_dedup.csv'] = csv_changes
print(f"  trip_filtered_events_dedup.csv: {len(csv_changes):,} CHANGE events")

# Build alarm_tag -> source file mapping (same logic as load_tag_events)
# tag_parquet_map already maps (tag, condition) -> parquet file path
# For CSV-only tags, the source is 'trip_filtered_events_dedup.csv'
def get_source_file(tag, condition):
    """Get the source file name for a given alarm tag."""
    if (tag, condition) in tag_parquet_map:
        return tag_parquet_map[(tag, condition)].name
    else:
        return 'trip_filtered_events_dedup.csv'

# Extract control actions for each alarm episode using SAME source
print("\nExtracting control actions from same source as alarm events...")
control_actions_list = []

for idx, alarm in alarms_significant.iterrows():
    source_file = get_source_file(alarm['tag'], alarm['condition'])
    source_changes = change_events_by_source[source_file]
    
    window_start = alarm['alarm_start'] - WINDOW_BEFORE
    window_end = alarm['alarm_end'] + WINDOW_AFTER
    
    # Get CHANGE events in this window from the SAME source
    mask = (source_changes['VT_Start'] >= window_start) & \
           (source_changes['VT_Start'] <= window_end)
    actions_in_window = source_changes[mask].copy()
    
    if len(actions_in_window) > 0:
        actions_in_window['alarm_tag'] = alarm['tag']
        actions_in_window['alarm_condition'] = alarm['condition']
        actions_in_window['alarm_start'] = alarm['alarm_start']
        actions_in_window['alarm_end'] = alarm['alarm_end']
        actions_in_window['alarm_idx'] = idx
        actions_in_window['source_file'] = source_file
        
        # Classify timing: before, during, after
        actions_in_window['timing'] = 'during'
        actions_in_window.loc[actions_in_window['VT_Start'] < alarm['alarm_start'], 'timing'] = 'before'
        actions_in_window.loc[actions_in_window['VT_Start'] > alarm['alarm_end'], 'timing'] = 'after'
        
        control_actions_list.append(actions_in_window)

control_actions_df = pd.concat(control_actions_list, ignore_index=True)
print(f"\nTotal control actions extracted: {len(control_actions_df):,}")
print(f"Unique operated tags (Source): {control_actions_df['Source'].nunique()}")
print(f"\nTiming breakdown:")
print(control_actions_df['timing'].value_counts())
print(f"\nSource file breakdown:")
print(control_actions_df['source_file'].value_counts())

Loading CHANGE events per source...
  02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet: 16,024 CHANGE events
  11266679-c683-445d-a1bf-29736e9e408f.parquet: 12,055 CHANGE events
  2c4de258-8026-4b49-8cfc-4a377becc808.parquet: 25,307 CHANGE events
  51fda50f-24a3-4632-9803-7ec34744c34b.parquet: 26,529 CHANGE events
  73601713-e81f-4f46-8631-062c4dcd22dc.parquet: 23,450 CHANGE events
  trip_filtered_events_dedup.csv: 115,857 CHANGE events

Extracting control actions from same source as alarm events...

Total control actions extracted: 55,510
Unique operated tags (Source): 91

Timing breakdown:
timing
after     20815
before    19459
during    15236
Name: count, dtype: int64

Source file breakdown:
source_file
02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet    21084
11266679-c683-445d-a1bf-29736e9e408f.parquet    15762
51fda50f-24a3-4632-9803-7ec34744c34b.parquet    15098
73601713-e81f-4f46-8631-062c4dcd22dc.parquet     3566
Name: count, dtype: int64


In [34]:
# ── Classify control action types from Description column ──────────
# Description contains values like 'OP', 'SP', 'MODE', 'SO', 'PVFL', etc.
def classify_action_type(desc):
    """Classify control action type from Description field."""
    if pd.isna(desc):
        return 'OTHER'
    desc_str = str(desc).strip().upper()
    if desc_str == 'OP':
        return 'OP'
    elif desc_str == 'SP':
        return 'SP'
    elif desc_str == 'MODE':
        return 'MODE'
    else:
        return 'OTHER'

control_actions_df['action_type'] = control_actions_df['Description'].apply(classify_action_type)

print("Control action type breakdown:")
print(control_actions_df['action_type'].value_counts())
print(f"\n{'='*70}")

# Filter to only OP, SP, MODE actions
control_actions_relevant = control_actions_df[
    control_actions_df['action_type'].isin(['OP', 'SP', 'MODE'])
].copy()
print(f"\nRelevant control actions (OP/SP/MODE): {len(control_actions_relevant):,} / {len(control_actions_df):,}")

# ── Which tags were operated most for each alarm tag? ──────────────
print("\n\nTop operated tags (Source) per alarm tag/condition:")
print("=" * 70)

for (alarm_tag, alarm_cond), grp in control_actions_relevant.groupby(['alarm_tag', 'alarm_condition']):
    n_alarms = len(alarms_significant[
        (alarms_significant['tag'] == alarm_tag) & (alarms_significant['condition'] == alarm_cond)
    ])
    print(f"\n{alarm_tag} | {alarm_cond} ({n_alarms} alarms, {len(grp)} OP/SP/MODE actions)")
    print("-" * 50)
    
    # Top operated tags with action type breakdown
    top_tags = grp.groupby(['Source', 'action_type']).size().unstack(fill_value=0)
    top_tags['total'] = top_tags.sum(axis=1)
    top_tags = top_tags.sort_values('total', ascending=False).head(10)
    print(top_tags.to_string())

Control action type breakdown:
action_type
OP       37149
SP       13973
MODE      4152
OTHER      236
Name: count, dtype: int64


Relevant control actions (OP/SP/MODE): 55,274 / 55,510


Top operated tags (Source) per alarm tag/condition:

03LIC_1016 | PVHI (154 alarms, 2067 OP/SP/MODE actions)
--------------------------------------------------
action_type  MODE    OP   SP  total
Source                             
03LIC_1016    162  1248  353   1763
03LIC_1034     23     3  101    127
03TIC_1009      2     0   45     47
03HIC_1023A     4    42    0     46
03PIC_1023     13     1   15     29
03TIC_1023     10     0   17     27
03HIC_1023B     1    13    0     14
03HIC_1009A     3     3    0      6
03HIC_1009B     3     3    0      6
03LIC_1031      0     0    2      2

03LIC_1016 | PVLO (2265 alarms, 6430 OP/SP/MODE actions)
--------------------------------------------------
action_type   MODE    OP    SP  total
Source                               
03LIC_1016     325  2229  1550   41

In [35]:
# ── Summary: action types and top operated tags across all alarm tags ───
print("Overall action type breakdown:")
print(control_actions_df['action_type'].value_counts())
print(f"\nRelevant (OP/SP/MODE): {len(control_actions_relevant):,}")
print(f"\n{'='*70}")
print("\nTop 15 most operated tags across ALL alarm episodes (OP/SP/MODE only):")
overall_top = control_actions_relevant.groupby(['Source', 'action_type']).size().unstack(fill_value=0)
overall_top['total'] = overall_top.sum(axis=1)
overall_top = overall_top.sort_values('total', ascending=False).head(15)
print(overall_top.to_string())

print(f"\n{'='*70}")
print("\nTiming breakdown for OP/SP/MODE actions:")
print(control_actions_relevant.groupby(['timing', 'action_type']).size().unstack(fill_value=0))

Overall action type breakdown:
action_type
OP       37149
SP       13973
MODE      4152
OTHER      236
Name: count, dtype: int64

Relevant (OP/SP/MODE): 55,274


Top 15 most operated tags across ALL alarm episodes (OP/SP/MODE only):
action_type  MODE     OP    SP  total
Source                               
03LIC_1619    978  10074  1831  12883
03LIC_1071    378   6817  1096   8291
03LIC_1016    639   4871  2357   7867
03TIC_1635    293   3272   742   4307
03LIC_1034    166    257  3212   3635
03FIC_1085    680   1974     0   2654
03PIC_1013     65   1690     0   1755
03FIC_1668      0   1562     0   1562
03LIC_1608     94    356   870   1320
03TIC_1009    236     21  1038   1295
03HIC_1151     93   1190     0   1283
03LIC_1085     67      0  1200   1267
03HIC_1092A     0   1225     0   1225
03SDV_1167      0    748     0    748
03PIC_1068     21     44   553    618


Timing breakdown for OP/SP/MODE actions:
action_type  MODE     OP    SP
timing                        
after        169

In [36]:
# Check timestamp ranges for each parquet file
from pathlib import Path
import pandas as pd

data_path = Path('/home/h604827/ControlActions/DATA/25_tags_events_preprocessed')
parquet_files = sorted(data_path.glob('*.parquet'))

print(f"{'File':<45} | {'Min Timestamp':<26} | {'Max Timestamp':<26} | {'Rows':>8}")
print("-" * 115)
for pf in parquet_files:
    df_tmp = pd.read_parquet(pf, columns=['VT_Start'])
    df_tmp['VT_Start'] = pd.to_datetime(df_tmp['VT_Start'])
    print(f"{pf.name:<45} | {str(df_tmp['VT_Start'].min()):<26} | {str(df_tmp['VT_Start'].max()):<26} | {len(df_tmp):>8,}")

File                                          | Min Timestamp              | Max Timestamp              |     Rows
-------------------------------------------------------------------------------------------------------------------
02e8226b-6cd8-43a3-b479-05a8dc9467bf.parquet  | 2021-10-02 00:05:35.153800 | 2025-06-28 02:55:18.589900 |  136,265
11266679-c683-445d-a1bf-29736e9e408f.parquet  | 2021-10-03 10:30:59.953200 | 2025-06-27 18:44:25.077400 |   43,675


2c4de258-8026-4b49-8cfc-4a377becc808.parquet  | 2021-10-02 23:22:02.091700 | 2025-06-28 01:57:27.900300 |   68,510
51fda50f-24a3-4632-9803-7ec34744c34b.parquet  | 2021-10-02 05:45:11.362000 | 2025-06-27 17:32:37.458400 |   70,213
73601713-e81f-4f46-8631-062c4dcd22dc.parquet  | 2021-09-06 12:33:15.353100 | 2025-06-27 18:21:07.686300 |  906,230
